# 04 · Calibration training

The one fitting step in this project (see `docs/adr/0001-no-fine-tuning.md` and `src/forecaster/calibration.py`). If your local database has 100+ resolved forecasts, this notebook will use your real data automatically; otherwise it falls back to simulated data so it still runs offline.

In [ ]:
import sys

sys.path.insert(0, "../src")
import numpy as np

from forecaster import calibration
from forecaster.scoring import brier_score, expected_calibration_error

In [ ]:
def simulate_biased_dataset(n, seed=0):
    rng = np.random.default_rng(seed)
    true_p = rng.uniform(0.05, 0.95, n)
    outcomes = (rng.random(n) < true_p).astype(float)
    raw = np.clip(0.5 + (true_p - 0.5) * 1.6, 0.02, 0.98)  # overconfident model
    timestamps = list(range(n))  # already in time order
    return raw, outcomes, timestamps


db_path = "../data/forecaster.db"
use_real_data = False  # flip to True once you have 100+ resolved forecasts
raw, outcomes, timestamps = simulate_biased_dataset(400)
print("using simulated data:", not use_real_data, "| n =", len(outcomes))

## Platt scaling vs isotonic regression on a temporal split

`temporal_split` fits on the earliest 70% of resolved forecasts and tests on the latest 30% — a random split would let the calibrator learn from questions that resolved *after* the ones it's tested on, which overstates how well it will do going forward.

In [ ]:
from datetime import datetime, timedelta

base = datetime(2026, 1, 1)
ts = [base + timedelta(days=i) for i in timestamps]
train_idx, test_idx = calibration.temporal_split(ts, train_frac=0.7)

for method in ["identity", "platt", "isotonic"]:
    cal = calibration.fit(raw[train_idx], outcomes[train_idx], method=method)
    test_calibrated = cal.predict(raw[test_idx])
    brier = brier_score(test_calibrated, outcomes[test_idx])
    ece = expected_calibration_error(test_calibrated, outcomes[test_idx])
    print(f"{method:10s} test Brier={brier:.4f}  test ECE={ece:.4f}")

## Auto method selection

`calibration.fit(..., method="auto")` chooses nothing below 50 resolved questions, Platt below 300, isotonic from 300 up — because isotonic regression needs more data to avoid overfitting its step function to noise.

In [ ]:
for n in [10, 100, 500]:
    print(n, "->", calibration.choose_method(n))

## Learning curve

How much training data each method needs before it reliably beats doing nothing.

In [ ]:
import matplotlib.pyplot as plt

sizes = [20, 40, 80, 150, 300, 600]
platt_briers, isotonic_briers, identity_briers = [], [], []
for size in sizes:
    r, o, t = simulate_biased_dataset(size, seed=42)
    ts2 = [base + timedelta(days=i) for i in t]
    tr, te = calibration.temporal_split(ts2, 0.7)
    for method, bucket in [
        ("identity", identity_briers),
        ("platt", platt_briers),
        ("isotonic", isotonic_briers),
    ]:
        cal = calibration.fit(r[tr], o[tr], method=method)
        bucket.append(brier_score(cal.predict(r[te]), o[te]))

plt.plot(sizes, identity_briers, "o-", label="identity (no correction)")
plt.plot(sizes, platt_briers, "o-", label="platt")
plt.plot(sizes, isotonic_briers, "o-", label="isotonic")
plt.xlabel("resolved forecasts used for fitting")
plt.ylabel("test Brier")
plt.legend()